# baseline — Python demo

Numerical companion to the entry [baseline](https://dictionaryofml.org/terms/baseline.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'baseline'. The GeoSphere Austria weather station Krems (station id 3805, 48.42 N, 15.62 E) records the air temperature every hour; this script downloads the records for 2024 from the GeoSphere data hub (resource klima-v2-1h), keeps the reading at 06:00 (UTC) and the one at 15:00 (UTC) of each day, and writes the 365 days that carry both to baseline_weather.csv, so the exact numbers behind the figure stay on record. Checks pin the downloaded values to the 2024 archive, so a change on the server side is caught rather than silently absorbed.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/baseline.py`](https://dictionaryofml.org/terms/baseline.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 baseline.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "baseline.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""A fundamental limit for the achievable loss, read off a weather record:
daily temperatures at Krems an der Donau, 2024.

Purpose
-------
Numerical companion to the glossary entry 'baseline'.  The GeoSphere
Austria weather station Krems (station id 3805, 48.42 N, 15.62 E)
records the air temperature every hour; this script downloads the
records for 2024 from the GeoSphere data hub (resource klima-v2-1h),
keeps the reading at 06:00 (UTC) and the one at 15:00 (UTC) of each day,
and writes the 365 days that carry both to baseline_weather.csv, so the
exact numbers behind the figure stay on record.  Checks pin the downloaded values to the 2024 archive, so a
change on the server side is caught rather than silently absorbed.

Each day is one data point: its feature is the 06:00 (UTC) reading, its
label the 15:00 (UTC) reading of the same day.  The 11 days whose 06:00
reading lies between 11.5 and 12.5 degrees C carry 15:00 readings from
12.4 to 28.4 degrees C: for (almost) the same feature value, the label
spreads over 16 degrees.  Every hypothesis
maps (almost) the same feature value to (almost) the same prediction,
so this spread bounds the achievable average loss from below.  The
strip-wise variance of the label, averaged over strips of the feature
axis that each hold at least four days, estimates that bound as 14.69
(squared degrees C); a hypothesis learned by linear regression reaches
an average squared error loss of 15.25 on the 365 days -- above the
estimate and within four percent of it.  The estimate is a floor only
for hypotheses that are constant on each strip (and, on the two days
with equal feature value, for every hypothesis), so for the linear
hypothesis the comparison is evidence rather than a guarantee: evidence
that little is left to gain.  That comparison against a baseline is exactly what the entry
is about.  A second block adds the baseline from the other side, the
one a method has to beat: predicting a single constant for every day,
which for a classification question means answering with the most
frequent label.

Deterministic: no randomness (the linear hypothesis is the closed-form
least-squares solution).  Self-contained: numpy + matplotlib only
(stdlib urllib for the download).

Blocks
------
[B-fetch]  Download the hourly air temperature at Krems for 2024 from
           the GeoSphere data hub and keep the 06:00 and the 15:00 (UTC)
           reading of each day; write the 365 records that carry both to
           baseline_weather.csv and check them against the archive.
[B-spread] The 11 days with a 06:00 reading in [11.5, 12.5): their 15:00
           readings range from 12.4 to 28.4 degrees C, and three days of
           the year share the exact feature value 0.7 with labels 1.0,
           5.8 and 17.3 -- no hypothesis can predict all three correctly.
           Write the band days and the remaining days to separate CSVs.
[B-limit] Estimate the smallest achievable average squared error loss
           by the strip-wise variance of the label (14.69).  The strips
           start from a 1-degree grid and are merged until each holds at
           least four days, so they vary in width and together cover
           every day.  Learn a linear hypothesis and check that its
           average squared error loss (15.25) sits above that estimate
           and within four percent of it -- a comparison, not a theorem,
           since a hypothesis may vary inside a strip.  Writes the strips for the
           entry's figure.
[B-trivial] The baseline from the other side: predicting one constant for
           every day.  For the squared error loss the best constant is
           the mean label (16.30 degrees C, average loss 87.23); for the
           classification question whether the 15:00 reading reaches 25
           degrees C, answering no every day is right on 283 of the 365
           days (accuracy 0.775).  The level rises with the rarity of
           the label: 0.915 for the 30-degree question.

Outputs
-------
baseline_weather.csv : date, t06, t15 of the 365 downloaded days
baseline_strips.csv  : lo, ymin, ymax of each strip of the feature axis
baseline_band.csv    : t06, t15 of the 11 days in the band [11.5, 12.5)
baseline_points.csv  : t06, t15 of the remaining 354 days
baseline_points_plot.csv : every 18th of those, the 20 dots of the figure
baseline_fit.csv     : t06, t15 along the learned linear hypothesis
baseline.png         : preview (checking only) -- the 365 data points,
                       the band, its label spread, and the learned
                       linear hypothesis
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []                         # collects (check name, pass/fail) pairs


def check(name, ok):                # records and prints one verification
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-fetch]** Download the hourly air temperature at Krems for 2024 from the GeoSphere data hub and keep the 06:00 and the 15:00 (UTC) reading of each day; write the 365 records that carry both to baseline_weather.csv and check them against the archive.

In [ ]:
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2024-01-01T00:00&end=2024-12-31T23:00")
with urllib.request.urlopen(URL, timeout=300) as resp:
    payload = json.load(resp)
readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
byday = {}
for stamp, value in zip(payload["timestamps"], readings):
    if value is None:                        # the archive drops whole days
        continue
    hour = stamp[11:16]                      # the stamps are UTC, and so are
    if hour in ("06:00", "15:00"):           # the two times
        byday.setdefault(stamp[:10], {})[hour] = float(value)
records = [(day, rec["06:00"], rec["15:00"])
           for day, rec in sorted(byday.items()) if len(rec) == 2]
with open(OUT_DIR / "baseline_weather.csv", "w") as f:
    f.write("date,t06,t15\n")
    for day, lo, hi in records:
        f.write(f"{day},{lo},{hi}\n")
check("[B-fetch] 365 of the days of 2024 carry both readings (the hourly "
      "record drops 2024-01-12)", len(records) == 365)
check("[B-fetch] no kept day is missing a reading",
      all(lo is not None and hi is not None for _, lo, hi in records))
check("[B-fetch] the record matches the archive (Jan 1: 4.0 at 06:00 and "
      "8.0 at 15:00, both UTC)",
      records[0] == ("2024-01-01", 4.0, 8.0))

**[B-spread]** The 11 days with a 06:00 reading in [11.5, 12.5): their 15:00 readings range from 12.4 to 28.4 degrees C, and three days of the year share the exact feature value 0.7 with labels 1.0, 5.8 and 17.3 -- no hypothesis can predict all three correctly. Write the band days and the remaining days to separate CSVs.

In [ ]:
temps = np.array([[lo, hi] for _, lo, hi in records])
t06, t15 = temps[:, 0], temps[:, 1]
band = (t06 >= 11.5) & (t06 < 12.5)

check("[B-spread] 11 days have their 06:00 reading in [11.5, 12.5)",
      int(band.sum()) == 11)
check("[B-spread] their 15:00 readings range from 12.4 to 28.4",
      round(float(t15[band].min()), 1) == 12.4
      and round(float(t15[band].max()), 1) == 28.4)
check("[B-spread] three days share the 06:00 reading 0.7, with 15:00 "
      "readings 1.0 / 5.8 / 17.3",
      sorted(t15[t06 == 0.7].tolist()) == [1.0, 5.8, 17.3])

header = "t06,t15"
np.savetxt(OUT_DIR / "baseline_band.csv", temps[band], delimiter=",",
           header=header, comments="", fmt="%.1f")
np.savetxt(OUT_DIR / "baseline_points.csv", temps[~band], delimiter=",",
           header=header, comments="", fmt="%.1f")
# the entry's figure draws every 18th of the 354 other days, 20 dots, so
# they do not hide the strips; the strips and every number use all 365
PLOT_EVERY = 18
np.savetxt(OUT_DIR / "baseline_points_plot.csv", temps[~band][::PLOT_EVERY],
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
check("[B-picture] the plotted subset of the other days holds 20 days",
      temps[~band][::PLOT_EVERY].shape[0] == 20)

**[B-limit]** Estimate the smallest achievable average squared error loss by the strip-wise variance of the label (14.69). The strips start from a 1-degree grid and are merged until each holds at least four days, so they vary in width and together cover every day. Learn a linear hypothesis and check that its average squared error loss (15.25) sits above that estimate and within four percent of it -- a comparison, not a theorem, since a hypothesis may vary inside a strip. Writes the strips for the entry's figure.

In [ ]:
# The strips start from a 1-degree grid and are merged with their right
# neighbour until each holds at least MIN_DAYS days, so every day enters
# exactly one strip and the sparse tails widen instead of being dropped.
MIN_DAYS = 4
grid = np.arange(-9.5, 27.5, 1.0)   # covers the 06:00 range, -8.8 to 26.2
strips, lo_edge, so_far = [], grid[0], 0
for k in range(len(grid) - 1):
    so_far += int(((t06 >= grid[k]) & (t06 < grid[k + 1])).sum())
    if so_far >= MIN_DAYS:
        strips.append((lo_edge, grid[k + 1]))
        lo_edge, so_far = grid[k + 1], 0
if so_far:                                   # absorb a short last strip
    strips[-1] = (strips[-1][0], grid[-1])

floor_sum, floor_days = 0.0, 0      # bound estimate: strip-wise variance
strip_rows = []
for lo, hi in strips:
    in_strip = (t06 >= lo) & (t06 < hi)
    n = int(in_strip.sum())
    floor_sum += float(t15[in_strip].var(ddof=1)) * n
    floor_days += n
    strip_rows.append((lo, float(t15[in_strip].min()),
                       float(t15[in_strip].max())))
floor = floor_sum / floor_days
strip_rows.append((strips[-1][1], strip_rows[-1][1], strip_rows[-1][2]))

check(f"[B-limit] every strip holds at least {MIN_DAYS} days",
      all(int(((t06 >= lo) & (t06 < hi)).sum()) >= MIN_DAYS
          for lo, hi in strips))
check("[B-limit] the strips partition the record: every day is in one",
      floor_days == len(t06))
check("[B-limit] the strips vary in width",
      len({round(hi - lo, 1) for lo, hi in strips}) > 1)

with open(OUT_DIR / "baseline_strips.csv", "w") as fh:
    fh.write("lo,ymin,ymax\n")
    for lo, ylo, yhi in strip_rows:
        fh.write(f"{lo:.1f},{ylo:.1f},{yhi:.1f}\n")

A = np.stack([t06, np.ones(len(t06))], 1)
w, *_ = np.linalg.lstsq(A, t15, rcond=None)
mse = float(np.mean((t15 - A @ w) ** 2))
print(f"  learned hypothesis: t15 = {w[0]:.3f} * t06 + {w[1]:.3f}")
print(f"  average squared error loss {mse:.2f}, "
      f"estimated bound {floor:.2f} (over {len(strips)} strips, "
      f"{floor_days} days)")

check("[B-limit] the bound estimate is 14.69 squared degrees C",
      round(floor, 2) == 14.69)
check("[B-limit] the learned hypothesis reaches average loss 15.25",
      round(mse, 2) == 15.25)
check("[B-limit] the learned hypothesis does not beat the bound, and comes "
      "within four percent of it", floor <= mse < 1.04 * floor)

t_line = np.array([t06.min(), t06.max()])
np.savetxt(OUT_DIR / "baseline_fit.csv",
           np.stack([t_line, w[0] * t_line + w[1]], 1), delimiter=",",
           header=header, comments="", fmt="%.3f")

# ---- outputs: preview PNG (checking only)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for lo, hi in strips:                       # the strips behind everything
    in_strip = (t06 >= lo) & (t06 < hi)
    ax.add_patch(plt.Rectangle(
        (lo, float(t15[in_strip].min())), hi - lo,
        float(t15[in_strip].max() - t15[in_strip].min()),
        facecolor="0.88", edgecolor="0.6", lw=0.4, zorder=0))
ax.plot([], [], "s", color="0.88", mec="0.6",
        label=f"strips of >= {MIN_DAYS} days, label range")
ax.plot(t06[~band], t15[~band], "o", color="gray", ms=3, mew=0,
        alpha=0.6, label="354 other days")
ax.plot(t06[band], t15[band], "^", mfc="none", mec="tab:red", ms=7,
        mew=1.5, label="11 days with the 06:00 reading in [11.5, 12.5)")
ax.plot(t_line, w[0] * t_line + w[1], "k-", lw=1.5,
        label="hypothesis learned by linear regression")
ax.annotate("", xy=(12.0, 28.4), xytext=(12.0, 12.4),
            arrowprops=dict(arrowstyle="<->", lw=1.2))
ax.text(13.5, 19.0, "spread of 16.0 °C\nat the same feature value",
        fontsize=8)
ax.set_xlabel("06:00 (UTC) reading of the day (feature, °C)")
ax.set_ylabel("15:00 (UTC) reading of the day (label, °C)")
ax.set_title("365 days at Krems, 2024: the label spread limits the loss")
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "baseline.png", dpi=150)

**[B-trivial]** The baseline from the other side: predicting one constant for every day. For the squared error loss the best constant is the mean label (16.30 degrees C, average loss 87.23); for the classification question whether the 15:00 reading reaches 25 degrees C, answering no every day is right on 283 of the 365 days (accuracy 0.775). The level rises with the rarity of the label: 0.915 for the 30-degree question.

In [ ]:
constant = float(t15.mean())
mse_constant = float(np.mean((t15 - constant) ** 2))
print(f"  predicting the constant {constant:.2f} degrees C on every day: "
      f"average squared error loss {mse_constant:.2f}, "
      f"{mse_constant / floor:.1f} times the estimated bound {floor:.2f}")
check("[B-trivial] the best constant is the mean label, 16.30 degrees C",
      round(constant, 2) == 16.30)
check("[B-trivial] its average squared error loss is the label variance, 87.23",
      round(mse_constant, 2) == 87.23
      and abs(mse_constant - float(t15.var())) < 1e-9)
check("[B-trivial] the learned hypothesis is four times better than that",
      mse < 0.25 * mse_constant)

for degrees in (25.0, 30.0):
    warm = t15 >= degrees                       # the label of the question
    share = float(warm.mean())
    accuracy = max(share, 1.0 - share)           # answering with the majority
    print(f"  is the 15:00 reading at least {degrees:.0f} degrees C? "
          f"{int(warm.sum())} of {len(warm)} days say yes; answering "
          f"{'yes' if share > 0.5 else 'no'} every day gives accuracy "
          f"{accuracy:.3f}, average zero-one loss {1.0 - accuracy:.3f}")
    if degrees == 25.0:
        acc25, warm25 = accuracy, warm
    else:
        acc30 = accuracy
check("[B-trivial] answering no on every day gets the 25-degree question "
      "right on 283 of the 365 days, an accuracy of 0.775",
      int((~warm25).sum()) == 283 and round(acc25, 3) == 0.775)
check("[B-trivial] the constant answer is right more often the rarer the "
      "label: 0.915 for the 30-degree question", round(acc30, 3) == 0.915)

# one split of the days on the feature, for contrast with the constant answer
splits = np.unique(t06)
scores = [( (t06 >= t_split).astype(int) == warm25.astype(int)).mean()
          for t_split in splits]
best = int(np.argmax(scores))
print(f"  one split of the days at a 06:00 reading of {splits[best]:.1f} "
      f"degrees C answers the 25-degree question with accuracy "
      f"{scores[best]:.3f}")
check("[B-trivial] a single split on the feature reaches accuracy 0.926, "
      "well above the constant answer",
      round(float(scores[best]), 3) == 0.926 and round(splits[best], 1) == 18.4)


failed = [name for name, ok in report if not ok]
print(f"{len(report) - len(failed)}/{len(report)} checks passed"
      + (f", FAILED: {failed}" if failed else ""))